In [1]:
# ===== কনফিগ: শুধু MODEL_NAME বদলিয়ে প্রতিবার চালান =====
# "swin_tiny" | "efficientnet_b0" | "convnext_tiny" | "resnet50" | "mobilenetv3_large"
MODEL_NAME = "swin_tiny"

import sys, time, copy, json, random, numpy as np, torch, torch.nn as nn, torch.optim as optim
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import DataLoader
from torchvision import models
sys.path.append("../src")
from dataset import gather_samples, compute_class_weights, RiceLeafDataset, CLASSES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_FOLDS, SEED, BATCH_SIZE, LR = 5, 42, 32, 1e-4
NUM_EPOCHS, PATIENCE, NUM_WORKERS = 30, 7, 4
SET_TORCH_SEED = True   # নতুন রানগুলো পুনরুৎপাদনযোগ্য করতে (fold অনুযায়ী SEED+fold)
OUT = Path("../results/cv"); OUT.mkdir(parents=True, exist_ok=True)

samples = gather_samples()
labels = np.array([s[1] for s in samples])
NUM_CLASSES = len(CLASSES)
print(device, MODEL_NAME, len(samples), NUM_CLASSES)

cuda swin_tiny 2753 7


In [3]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def make_model():
    if MODEL_NAME == "swin_tiny":
        m = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)
        m.head = nn.Linear(m.head.in_features, NUM_CLASSES)
    elif MODEL_NAME == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    elif MODEL_NAME == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_CLASSES)
    elif MODEL_NAME == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif MODEL_NAME == "mobilenetv3_large":
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, NUM_CLASSES)
    else:
        raise ValueError(MODEL_NAME)
    return m.to(device)

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, y in loader:
            imgs, y = imgs.to(device), y.to(device)
            if train: optimizer.zero_grad()
            out = model(imgs); loss = criterion(out, y)
            if train: loss.backward(); optimizer.step()
            loss_sum += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == y).sum().item(); total += y.size(0)
    return loss_sum / total, correct / total

def predict(model, loader):
    model.eval(); preds, gts = [], []
    with torch.no_grad():
        for imgs, y in loader:
            preds += model(imgs.to(device)).argmax(1).cpu().tolist(); gts += y.tolist()
    return gts, preds

def run_fold(fold, tr_idx, te_idx):
    if SET_TORCH_SEED: set_seed(SEED + fold)
    y_tr = labels[tr_idx]
    tr_i, va_i = train_test_split(tr_idx, test_size=0.1765, stratify=y_tr, random_state=SEED)
    tr_set = [samples[i] for i in tr_i]; va_set = [samples[i] for i in va_i]; te_set = [samples[i] for i in te_idx]

    mk = lambda ds, sp, sh: DataLoader(RiceLeafDataset(ds, split=sp), batch_size=BATCH_SIZE,
                                       shuffle=sh, num_workers=NUM_WORKERS, pin_memory=True)
    tl, vl, el = mk(tr_set, "train", True), mk(va_set, "val", False), mk(te_set, "test", False)

    model = make_model()
    w = torch.tensor(compute_class_weights(tr_set), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=w)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    best_loss, best_wts, bad = float("inf"), copy.deepcopy(model.state_dict()), 0
    for ep in range(NUM_EPOCHS):
        trl, tra = run_epoch(model, tl, criterion, optimizer)
        vl_loss, vl_acc = run_epoch(model, vl, criterion)
        scheduler.step(vl_loss)
        print(f"{MODEL_NAME} fold{fold} ep{ep+1} train_acc={tra:.4f} val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}")
        if vl_loss < best_loss:
            best_loss, best_wts, bad = vl_loss, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE: print("early stop"); break

    model.load_state_dict(best_wts)
    gts, preds = predict(model, el)
    del model; torch.cuda.empty_cache()
    return {"fold": fold,
            "macro_f1": f1_score(gts, preds, average="macro"),
            "acc": accuracy_score(gts, preds),
            "per_class_f1": f1_score(gts, preds, average=None, labels=list(range(NUM_CLASSES))).tolist(),
            "test_idx": [int(i) for i in te_idx], "gts": gts, "preds": preds}

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)   # 08 এর মতোই একই fold split
results = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), 1):
    t0 = time.time()
    r = run_fold(fold, tr_idx, va_idx); results.append(r)
    json.dump(results, open(OUT / f"{MODEL_NAME}_cv_results.json", "w"))
    print(f"== {MODEL_NAME} Fold {fold}: F1={r['macro_f1']:.4f} Acc={r['acc']:.4f} ({(time.time()-t0)/60:.1f} min)")

In [ ]:
f1s = np.array([r["macro_f1"] for r in results]); accs = np.array([r["acc"] for r in results])
pc = np.array([r["per_class_f1"] for r in results])
print(f"[{MODEL_NAME}] Macro-F1: {f1s.mean():.4f} ± {f1s.std():.4f} (pop) | ± {f1s.std(ddof=1):.4f} (sample)")
print(f"[{MODEL_NAME}] Accuracy: {accs.mean():.4f} ± {accs.std():.4f} (pop) | ± {accs.std(ddof=1):.4f} (sample)")
print("Fold F1:", [round(x, 4) for x in f1s])
for i, c in enumerate(CLASSES): print(f"{c:<16} {pc[:,i].mean():.3f} ± {pc[:,i].std(ddof=1):.3f}")